In [ ]:
from js import location
import micropip

_path = location.pathname
_root = _path.split("/extensions/")[0] if "/extensions/" in _path else _path.rsplit("/", 1)[0]
_pypi = location.origin + _root + "/pypi/"
_wheels = [
    'lark-1.3.1-py3-none-any.whl',
    'icontract-2.7.3-py3-none-any.whl',
    'asttokens-2.4.1-py2.py3-none-any.whl',
    'typing_extensions-4.16.0-py3-none-any.whl',
    'six-1.17.0-py2.py3-none-any.whl',
    'comm-0.2.3-py3-none-any.whl',
    'fpl_walker-0.0.0-py3-none-any.whl',
]
await micropip.install([_pypi + w for w in _wheels], deps=False)
import fpl
fpl


In [ ]:
import marimo as mo

# 01.04.02 Run

```
run = lex >> parse >> resolve >> expand >> elaborate >> lower >> evaluate
```

The executable chain is the prefix the walker actually runs. `desugar` stands in for resolve, expand, and lower.

`run_now` is a [PassCompose](01.04.01-pass.md): data, not a closure. This page
runs it with `direct`, in process. `COMPOSITIONS` names the compositions other
tools can find by name, so a command line or a graph view need not import them
one by one.

In [ ]:
from fpl.arrow import PassCompose, direct
from fpl.ast_core import Run
from fpl.desugar import desugar, listing
from fpl.elaborate import elaborate
from fpl.errors import FplError
from fpl.evaluate import evaluate
from fpl.lex import lex
from fpl.parse import parse
from fpl.print import render

run_now = lex >> parse >> desugar >> elaborate >> evaluate

COMPOSITIONS: dict[str, PassCompose[str, object]] = {"run": run_now}

INDENT_COLUMNS = 4


def _notebook_indent(source: str) -> str:
    """
    Turn leading spaces into tabs for notebook editors that indent with
    spaces.

    Uses ceiling division by INDENT_COLUMNS so two spaces still become
    one tab.
    """
    lines: list[str] = []
    for line in source.split("\n"):
        n = 0
        while n < len(line) and line[n] == " ":
            n += 1
        if n == 0:
            lines.append(line)
            continue
        levels = (n + INDENT_COLUMNS - 1) // INDENT_COLUMNS
        lines.append("\t" * levels + line[n:])
    return "\n".join(lines)


def _run(source: str, session: str) -> str:
    """
    Render one program. `session` is accepted and not stored.
    """
    del session
    return render(listing(direct(run_now, _notebook_indent(source))))


class Session:
    """
    A named run. The name is kept and not stored beyond the object.
    """

    def __init__(self, name: str) -> None:
        self.name = name

    def file(self, path: str) -> str:
        """
        Read a file as UTF-8 and render it.
        """
        source = open(path, encoding="utf-8").read()
        return _run(source, self.name)

    def src(self, source: str) -> str:
        """
        Render a source string.
        """
        return _run(source, self.name)


default = Session("default")


def run_file(path: str) -> int:
    """
    Run one file the way the seed walker does. Goals go to stderr.
    """
    try:
        text = default.file(path)
    except FplError as error:
        print(error)
        return 1
    print(text, end="")
    return 0